# Import

In [17]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import joblib

# Read the data and split the train and test memebers

In [ ]:
members_df = pd.read_csv('ikimina_members.csv')
groups_df = pd.read_csv('ikimina_groups.csv')
labels_df = pd.read_csv('labels.csv')

In [7]:
train_members = members_df[members_df['member_id'] <= 400].copy()
test_members = members_df[members_df['member_id'] > 400].copy()

In [8]:
print(members_df.columns)
print(groups_df.columns)

Index(['member_id', 'join_date', 'role', 'weekly_contrib_xaf',
       'penalty_paid_count', 'borrowed_total_xaf', 'repaid_total_xaf',
       'group_id', 'district', 'on_time_rate_m1', 'missed_count_m1',
       'on_time_rate_m2', 'missed_count_m2', 'on_time_rate_m3',
       'missed_count_m3', 'on_time_rate_m4', 'missed_count_m4',
       'on_time_rate_m5', 'missed_count_m5', 'on_time_rate_m6',
       'missed_count_m6', 'on_time_rate_m7', 'missed_count_m7',
       'on_time_rate_m8', 'missed_count_m8', 'on_time_rate_m9',
       'missed_count_m9', 'on_time_rate_m10', 'missed_count_m10',
       'on_time_rate_m11', 'missed_count_m11', 'on_time_rate_m12',
       'missed_count_m12'],
      dtype='object')
Index(['group_id', 'size', 'avg_contrib_xaf', 'founded_year', 'district',
       'urban_flag'],
      dtype='object')


# Feature Engineering

In [9]:
def engineer_features(members, groups):
    
    df = members.merge(groups, on='group_id', how='left') # merge the two on the group_id
    
    #Feature 1: Role Seniority (Ordinal encoding)
    role_map = {'member': 0, 'secretary': 1, 'treasurer': 1}
    df['feat_role_seniority'] = df['role'].map(role_map) #map the role into numbers based on the role_map
    
    #Feature 2: Tenure in months
    df['join_date'] = pd.to_datetime(df['join_date'])
    df['feat_tenure_months'] = (pd.Timestamp.now() - df['join_date']).dt.days // 30
    
    #Feature 3: Borrow-to-Repay Ratio
    # If borrowed is 0, the ratio is 1 . Otherwise, repaid / borrowed.
    df['feat_repayment_ratio'] = np.where(
        df['borrowed_total_xaf'] > 0,
        df['repaid_total_xaf'] / df['borrowed_total_xaf'],
        1.0
    )
    
    # Extract missed counts for the 12 months for vectorized operations
    missed_cols = [f'missed_count_m{i}' for i in range(1, 13)]
    missed_matrix = df[missed_cols].values
    
    # Feature 4: Total missed weeks
    df['feat_total_missed'] = df[missed_cols].sum(axis=1)
    
    # Feature 5: Recency-weighted miss rate 
    # Applies linearly increasing weights so recent misses (m12) hurt more than old ones (m1)
    weights = np.linspace(0.5, 2.0, 12) 
    df['feat_recent_miss_score'] = (missed_matrix * weights).sum(axis=1)
    
    # Feature 6: On-time streak (Max consecutive months with 0 misses)
    def get_max_streak(row):
        streak = max_streak = 0
        for val in row:
            if val == 0:
                streak += 1
                max_streak = max(max_streak, streak)
            else:
                streak = 0
        return max_streak
    
    df['feat_max_on_time_streak'] = df[missed_cols].apply(get_max_streak, axis=1)
    
    # Feature 7: Contribution relative to group average
    # Helps identify if they are overextending themselves compared to peers
    df['feat_contrib_vs_group'] = df['weekly_contrib_xaf'] / df['avg_contrib_xaf']
    
    # Feature 8: Penalty compliance rate
    df['feat_penalty_compliance'] = np.where(
        df['feat_total_missed'] > 0,
        df['penalty_paid_count'] / df['feat_total_missed'],
        1.0 
    )
    
    # Feature 9: Urban flag (Direct from group)
    df['feat_urban'] = df['urban_flag']
    
    # Filter down to just our engineered features and the member_id
    feature_cols = [col for col in df.columns if col.startswith('feat_')]
    return df[['member_id'] + feature_cols]

#Apply to train and test sets
X_train_raw = engineer_features(train_members, groups_df)
X_test_raw = engineer_features(test_members, groups_df)



# Train test split

In [10]:
#Merge with labels for the training set
train_data = X_train_raw.merge(labels_df, on='member_id') # add the labels to the tarining
y_train = train_data['defaulted_within_6m']
X_train = train_data.drop(columns=['member_id', 'defaulted_within_6m'])


X_test = X_test_raw.drop(columns=['member_id'])
y_test = labels_df[labels_df['member_id'] > 400]['defaulted_within_6m'] # used later for evaluation

In [11]:
print(X_train.columns,"********** \n", X_test.columns,"********** \n", y_train.name,"********** \n", y_test.name)

Index(['feat_role_seniority', 'feat_tenure_months', 'feat_repayment_ratio',
       'feat_total_missed', 'feat_recent_miss_score',
       'feat_max_on_time_streak', 'feat_contrib_vs_group',
       'feat_penalty_compliance', 'feat_urban'],
      dtype='object') ********** 
 Index(['feat_role_seniority', 'feat_tenure_months', 'feat_repayment_ratio',
       'feat_total_missed', 'feat_recent_miss_score',
       'feat_max_on_time_streak', 'feat_contrib_vs_group',
       'feat_penalty_compliance', 'feat_urban'],
      dtype='object') ********** 
 defaulted_within_6m ********** 
 defaulted_within_6m


# Traning XGBoost

In [15]:
from xgboost import XGBClassifier
import numpy as np

# Initialize the Model
model = XGBClassifier(
    n_estimators=50,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

# Train the model
model.fit(X_train, y_train)

# Predict probabilities
prob_default_test = model.predict_proba(X_test)[:, 1]

# Calculate the Reliability Index [0, 100]
# If the probability of default is 0.15 (15%), the reliability score becomes 85
reliability_scores = (1 - prob_default_test) * 100

# Map the Scores to the Required Tiers
def assign_tier(score):
    if score <= 40:
        return 'High Risk'
    elif score <= 70:
        return 'Watch'
    else:
        return 'Low Risk'

# Create the results DataFrame
test_results = X_test.copy()


test_results['actual_default'] = y_test.values 
test_results['prob_default'] = prob_default_test
test_results['reliability_score'] = np.round(reliability_scores).astype(int) # Round to nearest whole number
test_results['tier'] = test_results['reliability_score'].apply(assign_tier)

print("Model trained, Reliability Index calibrated, and NaNs fixed!")
test_results[['feat_total_missed', 'actual_default', 'prob_default', 'reliability_score', 'tier']].head(20)

Model trained, Reliability Index calibrated, and NaNs fixed!


,feat_total_missed,actual_default,prob_default,reliability_score,tier
0,0,0,0.007240,99,Low Risk
1,8,0,0.047853,95,Low Risk
2,3,0,0.007142,99,Low Risk
3,4,0,0.051275,95,Low Risk
4,2,0,0.002319,100,Low Risk
5,19,1,0.867412,13,High Risk
6,1,0,0.004331,100,Low Risk
7,11,0,0.090936,91,Low Risk
8,2,0,0.002972,100,Low Risk
9,3,0,0.002398,100,Low Risk


In [18]:
# Save the trained XGBoost model to a file
joblib.dump(model, 'ikimina_xgb_model.pkl')
print("Model saved successfully!")

Model saved successfully!
